<a href="https://colab.research.google.com/github/MishterBluesky/MemTorMD/blob/main/MemTorMD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MemTorMD

This colab environment has the ability to forms diverse lipid 'torroids'! **MemTorMD also allows you to insert proteins into torroidal division pores**, as well as in a periplasmic environment context of various heights. Choose the protein, the pore and the area per lipid. The programme will do the rest. Visualise your initial insertion here.

***Pores may be formed of either E.coli (inner membrane) or B. subtilis membranes***, however the lipid levels are modifiable within the script for custom species.

*If you dont need to insert proteins* and only want to study pores or other 3D architectures of membranes - use [TS2CG GUI A simpler version of MemTorMD we made for general use with other topologies such as vesicles](https://colab.research.google.com/drive/1Krjt9W5FlHZMGptuhVYpM9W7XRZC023e?usp=sharing)

For more control and for work specifically on toroid lipids use [ToroidMaker-basic
](https://colab.research.google.com/drive/1pL59-6Hsa51valFnuR6Z2GAScQPJ8vy4?usp=sharing)
ToroidMaker is heavily dependent on [TS2CG ](https://www.nature.com/articles/s41467-020-16094-y) a system which allows for pointilism of 3D structures and insertion of lipids along these structures. It adds to this base code, by overlaying a  pipeline made of new scripts that can count molecules created in TS2CG, boxing and morphing of custom 3D shapes in this case a pore, aswell as providing a protein insertion method by which simulations can be directly run. Using this method, reproducible pores of different sizes and multilayer lipid systems can be simulated with your protein(s) of interest or investigated for other properties. You may also view your output pore in browser. Inspired by [MemProtMD](https://colab.research.google.com/github/pstansfeld/MemProtMD/blob/main/MemProtMD_Insane.ipynb)


In [ ]:
#@title Install dependencies
%%capture
import os
import sys
import requests
from google.colab import files
!pip install py3dmol
!apt-get update && apt-get upgrade -y && apt-get install -y gzip pymol dssp
import py3Dmol
os.chdir('/content/')
!git clone https://github.com/timnugent/memembed
%cd memembed
!make
!git clone https://github.com/MishterBluesky/MemTorMD.git
%cd MemTorMD
!make
%cd ../
%cd ../
!pip install pdb2pqr vermouth gromacswrapper==0.8.3 MDAnalysis wget
!git clone https://github.com/pstansfeld/cg2at
!wget https://github.com/pstansfeld/MemProtMD/raw/main/gromacs.zip
!wget https://github.com/pstansfeld/MemProtMD/raw/main/gromacs_d.zip
!wget https://github.com/pstansfeld/MemProtMD/raw/main/martini_v300.zip
!wget https://github.com/MishterBluesky/MemTorMD.git
!unzip -o martini_v300.zip
!unzip -o gromacs_d.zip
!unzip -o MemTorMD-main.zip
%mv /content/usr/local/gromacs/ /usr/local/
!ln -s /usr/local/gromacs/bin/gmx /usr/bin/gmx
!ln -s /usr/local/gromacs/bin/gmx_d /usr/bin/gmx_d
import gromacs
import MDAnalysis
import glob
import shutil
import subprocess
import wget
import pandas as pd
from gromacs import grompp, editconf, make_ndx, trjconv, pdb2gmx, mdrun , mdrun_d

In [ ]:
#@title System Complexity- (Optional)
#@markdown If you would like to add your protein, or multiple proteins in two places, select this True, otherwise always False.
%%capture
!pip install py3dmol
!apt-get update && apt-get upgrade -y && apt-get install -y gzip pymol dssp
File_Location = "Upload"
Two_Proteins = "False" #@param ["True","False"]



In [ ]:
#@title Upload proteins
sys.path.append('/usr/local/lib/python3.7/site-packages/')

os.chdir('/content/')

if File_Location == "Upload":
    upload = files.upload()
    filename = next(iter(upload))
if Two_Proteins == "True":
    upload1 = files.upload()
    filename1 = next(iter(upload1))

def energy_minimize(filename, working_dir):
    if not os.path.exists("temp"):
        os.makedirs("temp")
    os.chdir("temp")
    os.system(f"pdb2pqr30 --ff CHARMM --keep-chain {working_dir + filename} pqr.pdb")

    pdb2gmx(f="pqr.pdb", ignh=True, ff="charmm27", water="tip3p", o="conf.pdb")
    editconf(f="conf.pdb", d=8, c=True, o="conf.pdb")

    with open("em.mdp", "w") as em:
        em.write("integrator = steep\nnsteps = 5000\nemtol = 100\nemstep = 0.001")

    grompp(f="em.mdp", maxwarn=5, o="em", c="conf.pdb")
    mdrun(deffnm="em", c="clean.pdb")

    trjconv(f="clean.pdb", o=working_dir + f"fixed-{filename}", s="em.tpr", input=("system"))

    os.chdir("..")
    shutil.rmtree("temp", ignore_errors=True)

def run_martinize2():
    cmd = [
        "martinize2",
        "-f", "protein.pdb",
        "-ff", "martini3001",
        "-x", "protein-cg.pdb",
        "-o", "protein-cg.top",
        "-elastic",
        "-ef", "500",
        "-eu", "1.0",
        "-el", "0.5",
        "-ea", "0",
        "-ep", "0",
        "-merge", "all",
        "-eunit", "chain",
        "-maxwarn", "100000",
        "-scfix"
    ]
    subprocess.run(cmd, check=True)

def run_martinize2_1():
    cmd = [
        "martinize2",
        "-f", "protein1.pdb",
        "-ff", "martini3001",
        "-x", "protein-cg1.pdb",
        "-o", "protein-cg1.top",
        "-elastic",
        "-ef", "500",
        "-eu", "1.0",
        "-el", "0.5",
        "-ea", "0",
        "-ep", "0",
        "-merge", "all",
        "-eunit", "chain",
        "-maxwarn", "100000",
        "-scfix"
    ]
    subprocess.run(cmd, check=True)


name = os.path.splitext(filename)[0]
working_dir = '/content/' + name + '/'
os.makedirs(working_dir, exist_ok=True)
os.rename(filename, working_dir + name + '.pdb')
os.chdir(working_dir)

def energy_minimize(filename, working_dir):
    if not os.path.exists("temp"):
        os.makedirs("temp")
    os.chdir("temp")
    os.system(f"pdb2pqr30 --ff CHARMM --keep-chain {working_dir + filename} pqr.pdb")

    pdb2gmx(f="pqr.pdb", ignh=True, ff="charmm27", water="tip3p", o="conf.pdb")
    editconf(f="conf.pdb", d=8, c=True, o="conf.pdb")

    with open("em.mdp", "w") as em:
        em.write("integrator = steep\nnsteps = 5000\nemtol = 100\nemstep = 0.001")

    grompp(f="em.mdp", maxwarn=5, o="em", c="conf.pdb")
    mdrun(deffnm="em", c="clean.pdb")

    trjconv(f="clean.pdb", o=working_dir + f"fixed-{filename}", s="em.tpr", input=("system"))

    os.chdir("..")
    shutil.rmtree("temp", ignore_errors=True)

energy_minimize(filename, working_dir)
if Two_Proteins == "True":
    os.chdir("..")
    name1 = os.path.splitext(filename1)[0]
    working_dir1 = '/content/' + name1 + '/'
    os.makedirs(working_dir1, exist_ok=True)
    os.rename(filename1, working_dir1 + name1 + '.pdb')
    os.chdir(working_dir1)
    energy_minimize(filename1, working_dir1)

def py3dmol_view(pdbfilename, working_dir):
    mview = py3Dmol.view(width=600, height=600)
    with open(working_dir + pdbfilename, "r") as f:
        mol1 = f.read()
    mview.addModel(mol1, "pdb")
    mview.setStyle({"cartoon": {"color": "spectrum"}})
    mview.setStyle({'resn': 'DUM'}, {'sphere': {}})
    mview.setStyle({'atom': 'BB'}, {'sphere': {}})
    mview.setStyle({'atom': 'PO4'}, {'sphere': {}})
    mview.setStyle({'atom': 'P'}, {'sphere': {}})
    mview.setBackgroundColor("0xffffff")
    mview.zoomTo()
    mview.show()

In [ ]:
#@title Predict Membrane Orientation
# @markdown This uses memembed to predict the proteins orientation in a flat membrane. This is then used to orient the protein in MemTorMD. This is useful, as the protein isnt always inserted correctly along the z axis. Have a peak at what is meant to look like.
working_dir = '/content/' + name + '/'
os.chdir(working_dir)
os.system(f'/content/{name}/fixed-{filename}')
print(name)
py3dmol_view(f'fixed-{filename}', working_dir)

if Two_Proteins == "True":
    print(name1)
    os.chdir(working_dir1)
    os.system(f'/content/memembed/bin/memembed -o memembed1.pdb fixed-{filename1}')
    py3dmol_view('memembed1.pdb', working_dir1)


os.chdir('/content/memembed/MemTorMD')
!chmod +x memdifference.sh
!./memdifference.sh /content/$name/fixed-$filename /content/output_file.txt

if Two_Proteins == "True":
    !./memdifference.sh /content/$name1/memembed1.pdb /content/output_file1.txt

# Read the content of the output file
with open('/content/output_file.txt', 'r') as file:
    output = file.readlines()
difference_z = float(output[-1].split(":")[-1].strip())
if Two_Proteins == "True":
    !./memdifference.sh /content/$name1/memembed1.pdb /content/output_file1.txt

    with open('/content/output_file1.txt', 'r') as file:
       output = file.readlines()
    print("Difference in Z-Position:", difference_z1)
    difference_z1 = float(output[-1].split(":")[-1].strip())
print("Difference in Z-Position:", difference_z)


SpoIIIEmemonly-poreshape


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Backbone COM: 134.450986
Difference in z-position: -119.45
Difference calculation complete. Result saved in /content/output_file.txt
Difference in Z-Position: -119.45


In [ ]:
#@title Set-up Coarse-Grained Membrane Protein System
#@markdown This simplifies your protein(s) so it/they can be simulated in a large system such as the pore!
%%capture
os.chdir(working_dir)
!cp fixed-$filename memembed.pdb
for file in glob.glob(r'/content/martini_v300/martini*.itp'):
    print(file)
    shutil.copy(file, working_dir)

make_ndx(f='memembed.pdb', o='index.ndx', input=('del 0', 'del 1-100','rDUM','q'),backup=False)
editconf(f='memembed.pdb',o='protein.pdb',n='index.ndx',c=True,d=3,input=(0,0),backup=False)

u = MDAnalysis.Universe('protein.pdb')


box = (u.dimensions/10.0)

with open('protein.pdb', 'r') as file :
    filedata = file.read()
filedata = filedata.replace('HSE', 'HIS').replace('HSD', 'HIS').replace('MSE', 'MET').replace(' SE ', ' SD ')
with open('protein.pdb', 'w') as file:
    file.write(filedata)

run_martinize2()
if Two_Proteins == "True":
    os.chdir(working_dir1)

    for file in glob.glob(r'/content/martini_v300/martini*.itp'):
        print(file)
        shutil.copy(file, working_dir1)

    make_ndx(f='memembed1.pdb', o='index1.ndx', input=('del 0', 'del 1-100','rDUM','q'),backup=False)
    editconf(f='memembed1.pdb',o='protein1.pdb',n='index1.ndx',c=True,d=3,input=(0,0),backup=False)

    u = MDAnalysis.Universe('protein1.pdb')


    box = (u.dimensions/10.0)

    with open('protein1.pdb', 'r') as file :
       filedata1 = file.read()
    filedata = filedata.replace('HSE', 'HIS').replace('HSD', 'HIS').replace('MSE', 'MET').replace(' SE ', ' SD ')
    with open('protein1.pdb', 'w') as file:
        file.write(filedata1)

    run_martinize2_1()

## **MemTorMD**

In [ ]:
#@markdown In this section, you choose your pore size in nanometers. A typical division pore might be between 40nm to 20nm including membranes, and at least 5nm across to accomodate two DNA helices. You may however want almost or open closed conformations. Torroids also exist at other cell regions such as phagocytic end sites, eg: engulfment For these, pores might want to be more than 20nm across or more.
%%capture
Pore_type= 'Normal'  #@param {string} ['Normal', 'No pore', 'single membrane']
membrane_list = {
    "Normal": '3_Bigcare.tsi',
    "No pore": '3_Doublemembrane.tsi',
    "single membrane": '3_singlemembrane.tsi'
}
tsi = membrane_list.get(Pore_type)
%cd /content/memembed/MemTorMD
import math
from math import sqrt
import math
def scale_coordinates(input_file, output_file, scale_factor_x, scale_factor_y, scale_factor_z):
    with open(input_file, 'r') as infile:
        lines = infile.readlines()

    scaled_vertices = []
    stop_processing = False
    for line in lines:
        parts = line.split()
        if len(parts) > 1:
            if parts[0] == 'triangle':
                stop_processing = True
            if not stop_processing:
                if parts[0] == 'box':
                    # Scale box dimensions
                    box_dimensions = [float(dim) for dim in parts[1:]]
                    scaled_box_dimensions = [
                        box_dimensions[0] * scale_factor_x,
                        box_dimensions[1] * scale_factor_y,
                        box_dimensions[2] * scale_factor_z
                    ]
                    scaled_box_line = f"box\t{scaled_box_dimensions[0]}\t{scaled_box_dimensions[1]}\t{scaled_box_dimensions[2]}\n"
                    scaled_vertices.append(scaled_box_line)
                elif parts[0].isdigit():
                    vertex_id = int(parts[0])
                    x = float(parts[1]) * scale_factor_x
                    y = float(parts[2]) * scale_factor_y
                    z = float(parts[3]) * scale_factor_z
                    scaled_vertex = f"{vertex_id}\t{x}\t{y}\t{z}\n"
                    scaled_vertices.append(scaled_vertex)
                else:
                    scaled_vertices.append(line)
            else:
                scaled_vertices.append(line)

    with open(output_file, 'w') as outfile:
        for vertex in scaled_vertices:
            outfile.write(vertex)
#@markdown Pore and environment size in nanometers
pore_size = 5 #@param  # New pore size in nm
original_pore_height = 20
pore_height = 20 #@param
original_pore_size = 4
# Calculate scaling factors
scale_factor_y = math.sqrt(pore_size / original_pore_size)
scale_factor_z = math.sqrt(pore_size / original_pore_size)  # Assuming z scaling is the same as x and y
scale_factor_x = math.sqrt(pore_height / original_pore_height)  # Scale factor for x dimension of the box

# Scale the coordinates using the calculated scaling factor
scale_coordinates(tsi, "output.tsi", scale_factor_x, scale_factor_y, scale_factor_z)
tsi = "output.tsi"
#@markdown Advanced options
#@markdown , if you require a custom .tsi label it modified.tsi
customtsi = "False" #@param ["True","False"]
if customtsi == "True":
    uploadtsi = files.upload()
    tsiname = next(iter(uploadtsi))
APL= "0.65" #@param {numerical} [0.57,0.60,0.62,0.63,0.65,0.67,0.70,0.65]
Species = "Escherichia coli" #@param ["Escherichia coli", "Bacillus subtilis"]
if Species == "Bacillus subtilis":
      cardio = 0.10
      popg = 0.70
      pope = 0.20
else:
      cardio = 0.02
      popg = 0.20
      pope = 0.78

In [ ]:
#@title Insert proteins and create your toroid!
#@markdown ToroidMaker is a custom script that uses TS2CG, as well as a .tsi file which we have altered above, to create a pore. It then uses the martini protein with its membrane setup established, to insert proteins of interest into that pore.
%%capture
%cd /content/memembed/MemTorMD
!chmod +x proteinaligncolab.sh
!chmod +x martinishcolab.sh
!chmod +x topollipidp.sh
!chmod +x compile.sh
!chmod +x tsimodifier.sh
!chmod +x tsimodifiermulti.sh
!chmod +x proteinaligncolabmulti.sh
!chmod +x martinishmulti.sh
!chmod +x topollipidpmulti.sh
!chmod +x CustomLarge.sh
if Two_Proteins == "True":
    name1 = os.path.splitext(filename1)[0]
poredir = "/content/memembed/MemTorMD"
workdir = '/content/' + name + '/'
if Two_Proteins == "True":
    workdir1 = '/content/' + name1 + '/'
!gmx editconf -f "/content/{name}/protein-cg.pdb" -o "/content/memembed/MemTorMD/{name}.gro"
if Two_Proteins == "True":
    !gmx editconf -f "/content/{name1}/protein-cg1.pdb" -o "/content/memembed/MemTorMD/{name1}.gro"
protein_pdb = f"{name}.gro"
if Two_Proteins == "True":
    protein_pdb1 = f"{name1}.gro"
name = f"{name}"
if Two_Proteins == "True":
    name1 = f"{name1}"
verbal_list = {
    "Flat region SU": 159,
    "Just outside pore SU": 143,
    "Outer pore SU": 48,
    "Inner pore SU": 80,
    "Pore centre": 191,
    "Inner pore SD": 217,
    "Outer pore SD": 233,
    "Just Outside pore SD": 264,
    "Flat region SD": 128,
    "Centre - Side 1 No pore": 31,
    "Centre - Side 2 No pore": 124,
    "OffCentre - Side 1 No pore": 85,
    "OffCentre - Side 1 No pore": 329
}
verbal_list1 = {
    "Flat region SU": 159,
    "Just outside pore SU": 143,
    "Outer pore SU": 48,
    "Inner pore SU": 80,
    "Pore centre": 191,
    "Inner pore SD": 217,
    "Outer pore SD": 233,
    "Just Outside pore SD": 264,
    "Flat region SD": 128,
    "Centre - Side 1 No pore": 31,
    "Centre - Side 2 No pore": 124,
    "OffCentre - Side 1 No pore": 85,
    "OffCentre - Side 1 No pore": 329
}

#@markdown You may flip the molecule if its the wrong way, and alter the area per lipid. Moving it along the Z axis may be necessary, however it should be oriented correctly.
Flip = 'True' #@param ['True','False']
if Flip == "True":
    !gmx editconf -f "{name}.gro" -o "{name}.gro" -rotate 0 87 0
    print("Rotation applied.")
else:
    print("No rotation applied.")
#@markdown You may also translate the molecule slightly along the Z axis (remember a membrane is 4.3nm deep)
adjust1 = 0 #@param
Translate_Z_Nanometers = difference_z/10 + adjust1
if Flip == "True":
    Translate_Z_Nanometers = -Translate_Z_Nanometers
#@markdown **For your SECOND protein** (If applicable - else this does nothing)
Flip_2 = 'False' #@param ['True','False']
if Flip_2 == "True":
    !gmx editconf -f "{name1}.gro" -o "{name1}.gro" -rotate 0 180 0
    print("Rotation applied.")
else:
    print("No rotation applied.")
adjust2 = 0 #@param
if Two_Proteins =="True":
    Translate_Z_Nanometers_2 = difference_z1/10 + adjust2
    if Flip_2 == "True":
        Translate_Z_Nanometers_2 = -Translate_Z_Nanometers_2
#@markdown The location of your protein(s)
Location_PROTEIN1 = "Pore centre" #@param ["Flat region SU","Just outside pore SU","Outer pore SU","Inner pore SU","Pore centre","Inner pore SD","Outer pore SD","Just Outside pore SD","Flat region SD", "Centre - Side 1 No pore", "Centre - Side 2 No pore", "OffCentre - Side 1 No pore","OffCentre - Side 1 No pore"]
#@markdown (if applicable- otherwise does nothing)
Location_PROTEIN2 = "Just Outside pore SD" #@param ["Flat region SU","Just outside pore SU","Outer pore SU","Inner pore SU","Pore centre","Inner pore SD","Outer pore SD","Just Outside pore SD","Flat region SD", "Centre - Side 1 No pore", "Centre - Side 2 No pore", "OffCentre - Side 1 No pore","OffCentre - Side 1 No pore"]
# Look up the corresponding variable based on the chosen item
vertex = verbal_list.get(Location_PROTEIN1)
vertex1 = verbal_list1.get(Location_PROTEIN2)
if Two_Proteins == "True":
    protein_no = 2
else:
    protein_no = 1
if Two_Proteins =="True":
    second = name1
    vortex = vertex1
    workdsec =workdir1
    proteinaceous = protein_pdb1


if customtsi == "True":

    command = f"./CustomLarge.sh '{APL}' '{name}' '{tsi}' '{protein_pdb}' '{protein_no}' '{vertex}' '{workdir}' '{poredir}' '{Translate_Z_Nanometers}' "
else:
    if Two_Proteins == "True":
       command = f"./proteinaligncolabmulti.sh '{APL}' '{name}' '{tsi}' '{protein_pdb}' '{protein_no}' '{vertex}' '{workdir}' '{poredir}' '{proteinaceous}' '{second}' '{workdsec}' '{vortex}' '{Translate_Z_Nanometers}' '{Translate_Z_Nanometers_2}' '{scale_factor_z}' '{scale_factor_x}' '{cardio}' '{popg}' '{pope}'"
    else:
       command = f"./proteinaligncolab.sh '{APL}' '{name}' '{tsi}' '{protein_pdb}' '{protein_no}' '{vertex}' '{workdir}' '{poredir}' '{Translate_Z_Nanometers}' '{scale_factor_z}' '{scale_factor_x}' '{cardio}' '{popg}' '{pope}'"
!{command}

In [ ]:
#@title Visualise Pore with protein inserted. (Pre-equillibration)
working_dir = "/content/memembed/MemTorMD/"
def py3dmol_view(pdbfilename, working_dir):
    mview = py3Dmol.view(width=1200, height=1200)
    with open(working_dir + pdbfilename, "r") as f:
        mol1 = f.read()
    mview.addModel(mol1, "pdb")
    mview.setStyle({"cartoon": {"color": "spectrum"}})
    mview.setStyle({'atom': 'CB'}, {'sphere': {"color": "green"}})
    mview.setStyle({'atom': 'CA'}, {'sphere': {"color": "green"}})
    mview.setStyle({'atom': 'CG'}, {'sphere': {"color": "green"}})
    mview.setStyle({'atom': 'BB'}, {'sphere': {"color": "green"}})
    mview.setStyle({'atom': 'PO4'}, {'sphere': {}})
    mview.setStyle({'atom': 'P'}, {'sphere': {}})
    mview.setBackgroundColor("0xffffff")
    mview.zoomTo()
    mview.show()
APLFILE = f"7_Lipidx_{name}.gro"
!echo $APL
!gmx editconf -f $APLFILE -o "output.pdb"
py3dmol_view("output.pdb",working_dir)

0.65
                     :-) GROMACS - gmx editconf, 2023.4 (-:

Executable:   /usr/local/gromacs/bin/gmx
Data prefix:  /usr/local/gromacs
Working dir:  /content/memembed/MemTorMD
Command line:
  gmx editconf -f 7_Lipidx_rodapbp2ecoli.gro -o output.pdb

Note that major changes are planned in future for editconf, to improve usability and utility.
Read 618400 atoms
Volume: 623327 nm^3, corresponds to roughly 280497300 electrons
No velocities found

GROMACS reminds you: "Come and play on the hospital roof, I got something that's yours" (Sherlock)

Buffered data was truncated after reaching the output size limit.

In [ ]:
#@markdown Finally ToroidMaker will analyse your gro file and return a topol.top. You can use this and the associated .itp files as well as .gro file to run a simulation.
if Two_Proteins == "False":
    topol = f"./topollipidp.sh  '{APLFILE}' '{name}' '{protein_no}'"
else:
    topol = f"./topollipidpmulti.sh  '{APLFILE}' '{name}' '{protein_no}' {name1}"
# Execute the topol
!{topol}

Okay lipid person, I am reading the .gro file you gave me and adding POPG to the topol file now
Okay lipid person, I am reading the .gro file you gave me and adding POPE to the topol file now
Okay lipid person, I am reading the .gro file you gave me and adding CDL1 to the topol file now
Okay lipid person, I am reading the .gro file you gave me and adding POPG to the topol file now
Okay lipid person, I am reading the .gro file you gave me and adding POPE to the topol file now
Okay lipid person, I am reading the .gro file you gave me and adding CDL1 to the topol file now


In [ ]:
#@title Download files for simulation
import os
from google.colab import files

# List of files to zip
files_to_zip = ['martini_v3.0.0.itp', 'martini_v3.0_CDLs.itp', 'martini_v3.0.0_ions_v1.itp', 'martini_v3.0.0_solvents_v1.itp', 'martini_v3.0.0_phospholipids_v1.itp','topol.top',f'{APLFILE}']

# Additional .itp files that don't include the word "martini"
additional_itp_files = [filename for filename in os.listdir('.') if filename.endswith('.itp') and 'martini' not in filename]

zip_file_name = f'MemTorMD_{APL}_{pore_size}nm-pore_{pore_height}nm_layers.zip'

files_to_zip += additional_itp_files

os.system('zip -r {} {}'.format(zip_file_name, ' '.join(map(str, files_to_zip))))

files.download(zip_file_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#@title Simulate! (for 2000 steps).. and download a tpr for a real simulation. (or to check it simulates with gmx_d) - Please note that you must choose a suitable ion concentration for custome scenarios manually
!chmod +x ProteinAfterColab.sh
Simulate = f"./ProteinAfterColab.sh '{APL}' '{name}'"
# Execute Sim
!{Simulate}

Streaming output truncated to the last 5000 lines.
Replacing solvent molecule 1544493 (atom 2162893) with NA
Replacing solvent molecule 1751542 (atom 2369942) with NA
Replacing solvent molecule 3159913 (atom 3778313) with NA
Replacing solvent molecule 190861 (atom 809261) with NA
Replacing solvent molecule 3360699 (atom 3979099) with NA
Replacing solvent molecule 2383451 (atom 3001851) with NA
Replacing solvent molecule 4163508 (atom 4781908) with NA
Replacing solvent molecule 1706077 (atom 2324477) with NA
Replacing solvent molecule 1538312 (atom 2156712) with NA
Replacing solvent molecule 1499789 (atom 2118189) with NA
Replacing solvent molecule 4447543 (atom 5065943) with NA
Replacing solvent molecule 553639 (atom 1172039) with NA
Replacing solvent molecule 763650 (atom 1382050) with NA
Replacing solvent molecule 1941135 (atom 2559535) with NA
Replacing solvent molecule 2503554 (atom 3121954) with NA
Replacing solvent molecule 3084182 (atom 3702582) with NA
Replacing solvent molecul

In [ ]:
#@title Download files for simulation
import os
from google.colab import files
APLFILE2 = f"11b_Lipid20_{name}.gro"
TPR = f"12_Lipid20_{name}.tpr"
# List of files to zip
files_to_zip = ['martini_v3.0.0.itp', 'martini_v3.0_CDLs.itp', 'martini_v3.0.0_ions_v1.itp', 'martini_v3.0.0_solvents_v1.itp', 'martini_v3.0.0_phospholipids_v1.itp','topol.top',f'{APLFILE}',f'{APLFILE2}',f'{TPR}']

# Additional .itp files that don't include the word "martini"
additional_itp_files = [filename for filename in os.listdir('.') if filename.endswith('.itp') and 'martini' not in filename]

# Destination zip file name
zip_file_name = f'MemTorMD_{APL}_{pore_size}nm-pore_{pore_height}nm_layers.zip'

# Combine the lists of files to zip
files_to_zip += additional_itp_files

# Zip the files
os.system('zip -r {} {}'.format(zip_file_name, ' '.join(map(str, files_to_zip))))

# Download the zip file
files.download(zip_file_name)

In [ ]:
APLFILE2 = f"11b_Lipid20_{name}.gro"
!gmx editconf -f $APLFILE2 -o "equilibrated.pdb"
def py3dmol_view(pdbfilename, working_dir):
    mview = py3Dmol.view(width=1200, height=1200)
    with open(working_dir + pdbfilename, "r") as f:
        mol1 = f.read()
    mview.addModel(mol1, "pdb")
    mview.setStyle({'atom': 'CB'}, {'sphere': {"color": "green"}})
    mview.setStyle({'atom': 'CA'}, {'sphere': {"color": "green"}})
    mview.setStyle({'atom': 'CG'}, {'sphere': {"color": "green"}})
    mview.setStyle({'atom': 'BB'}, {'sphere': {"color": "green"}})
    mview.setStyle({'atom': 'NA'}, {'sphere': {"color": "blue"}})
    mview.setStyle({'atom': 'PO4'}, {'sphere': {}})
    mview.setStyle({'atom': 'P'}, {'sphere': {}})
    mview.setBackgroundColor("0xffffff")
    mview.zoomTo()
    mview.show()
py3dmol_view(f"equilibrated.pdb",working_dir)